## Homework 10 - Spark Structured Streaming
Jacob A. Fericy

Apologies for lack of explanation, running tight on deadlines.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, col, expr
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

import os
import shutil
import time

spark = SparkSession.builder.appName("HW10_Fericy").getOrCreate()
spark

## Optional file check

This cell checks that the six bike CSV files are in the current working directory.  
If you uploaded them into JupyterHub next to this notebook, these should all return `True`.

In [ ]:
required_files = [
    "bikeDetails_for_fit.csv",
    "bikeDetails_add1.csv",
    "bikeDetails_add2.csv",
    "bikeDetails_add3.csv",
    "bikeDetails_add4.csv",
    "bikeDetails_add5.csv",
]

for f in required_files:
    print(f, "->", os.path.exists(f))

# Part 1 - Streaming data using `rate`

Need the `rate` format, then two transformations on the `value` column:

- square root of `value`
- `vaule mod 4`

Then the result should be written to **memory** with a query name, allowed to run for about 30 seconds, stopped, and queried back with `spark.sql(...)`.

## Create the rate stream

In [ ]:
rateDF = spark.readStream.format("rate").load()
rateDF.printSchema()
rateDF

## Apply the required transformations
Need to create the stream first, then apply DataFrame transformations before starting the writer. 

In [ ]:
part1DF = (
    rateDF
    .withColumn("sqrt_value", sqrt(col("value")))
    .withColumn("mod_4", expr("value % 4"))
)

part1DF

## Write the transformed stream to memory and start the query

We use `format("memory")` and a query name so the results can be queried with Spark SQL after the stream is stopped.

In [ ]:
part1_query = (
    part1DF.writeStream
    .format("memory")
    .queryName("hw10_rate_table")
    .outputMode("append")
    .start()
)

print("Part 1 query started.")

## Let the stream run for about 30 seconds, then stop it

In [ ]:
time.sleep(30)

part1_query.stop()
print("Part 1 query stopped.")

## Query the in-memory table

In [ ]:
spark.sql("SELECT * FROM hw10_rate_table").show(truncate=False)

# Part 2 - CSV data with a Pipeline

## Read the training / fit CSV as a Spark DataFrame

In [ ]:
bike_fit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("bikeDetails_for_fit.csv")
)

bike_fit_df.show(5, truncate=False)
bike_fit_df.printSchema()

## Define the SQLTransformer
It log-transforms the response and mileage variable, keeps `year`, and creates a 0/1 indicator for first-owner bikes.

In [ ]:
sql_transformer = SQLTransformer(statement="""
SELECT
    log(selling_price) AS label,
    year,
    log(km_driven) AS log_km_driven,
    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
""")

## Define the VectorAssembler
The `features` column must contain:
- `year`
- `log_km_driven`
- `one_owner`

In [ ]:
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

## Create and fit the pipeline
1. `SQLTransformer`
2. `VectorAssembler`

In [ ]:
pipeline = Pipeline(stages=[sql_transformer, assembler])
pipeline_model = pipeline.fit(bike_fit_df)

print("Pipeline fitted successfully.")

## Apply the fitted pipeline to the static fit data once
This is just a quick verification step before using the same fitted pipeline on streaming input.

In [ ]:
bike_fit_transformed = pipeline_model.transform(bike_fit_df)
bike_fit_transformed.select("label", "year", "log_km_driven", "one_owner", "features").show(5, truncate=False)

## Create a streaming input folder
This cell creates empty directory.

In [ ]:
stream_dir = "bike_stream_input"

if os.path.exists(stream_dir):
    shutil.rmtree(stream_dir)

os.makedirs(stream_dir, exist_ok=True)

print("Streaming input folder ready:", stream_dir)
print("Current contents:", os.listdir(stream_dir))

## Set up the streaming CSV reader
- schema is required for `readStream`
- schema can be taken from the static DataFrame created above
- each CSV has a header row

In [ ]:
bike_stream_df = (
    spark.readStream
    .schema(bike_fit_df.schema)
    .option("header", True)
    .csv(stream_dir)
)

bike_stream_df.printSchema()

## Apply the fitted pipeline to the streaming DataFrame

In [ ]:
bike_stream_transformed = pipeline_model.transform(bike_stream_df)

bike_stream_transformed.select(
    "label", "year", "log_km_driven", "one_owner", "features"
)

## Write the transformed stream to the console

In [ ]:
part2_query = (
    bike_stream_transformed.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .start()
)

print("Part 2 query started.")

## Copy the five add-files into the watched folder one at a time

In [ ]:
add_files = [
    "bikeDetails_add1.csv",
    "bikeDetails_add2.csv",
    "bikeDetails_add3.csv",
    "bikeDetails_add4.csv",
    "bikeDetails_add5.csv",
]

for f in add_files:
    destination = os.path.join(stream_dir, f)
    shutil.copy(f, destination)
    print(f"Copied {f} -> {destination}")
    time.sleep(5)

## Stop the streaming query after all five files have been processed

In [ ]:
part2_query.stop()
print("Part 2 query stopped.")